# Fase 2 (paso 2) — Entrenamiento y evaluación de YOLOv8

## Configuración
- Dataset: `dataset_split/` con split original de Roboflow (198 train / 8 valid / 8 test, 5 clases).
- Modelo: **YOLOv8n** preentrenado en COCO.
- 200 épocas, `patience=50`, `imgsz=640`, `batch=8`.
- Augmentations conservadoras: HSV moderado, fliplr, sin mosaic ni mixup (perjudican datasets pequeños con escenas heterogéneas).
- Dispositivo: auto (MPS si Apple Silicon) con `PYTORCH_ENABLE_MPS_FALLBACK=1`.

## Salidas
- `runs/detect/coins_v4/` — pesos, métricas, gráficos generados por Ultralytics.
- `model/best.pt` — copia del mejor checkpoint para uso posterior.
- `reports/metrics_summary.csv`, `reports/metrics_per_class_valid.csv`.

In [1]:
from __future__ import annotations
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from ultralytics import YOLO

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_YAML = ROOT / "dataset_split" / "data.yaml"
MODEL_DIR = ROOT / "model"; MODEL_DIR.mkdir(exist_ok=True)
REPORTS = ROOT / "reports"; REPORTS.mkdir(exist_ok=True)
RUNS_DIR = ROOT / "runs"
RUN_NAME = "coins_v4"

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"DATA   = {DATA_YAML}")
print(f"DEVICE = {DEVICE}")
print(f"RUN    = {RUNS_DIR / 'detect' / RUN_NAME}")

DATA   = /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/dataset_split/data.yaml
DEVICE = mps
RUN    = /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/runs/detect/coins_v4


## 1. Entrenamiento

In [2]:
prev = RUNS_DIR / "detect" / RUN_NAME
if prev.exists():
    shutil.rmtree(prev)

model = YOLO("yolov8n.pt")
results = model.train(
    data=str(DATA_YAML),
    epochs=200,
    patience=50,
    imgsz=640,
    batch=8,
    device=DEVICE,
    project=str(RUNS_DIR / "detect"),
    name=RUN_NAME,
    exist_ok=True,
    seed=42,
    deterministic=True,
    plots=True,
    verbose=True,
    # Augmentations conservadoras para dataset pequeño y heterogéneo
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    degrees=5.0, translate=0.05, scale=0.3,
    fliplr=0.5, flipud=0.0,
    mosaic=0.0, mixup=0.0,
    erasing=0.0,
)
print("Entrenamiento finalizado.")

Ultralytics 8.4.47 🚀 Python-3.12.7 torch-2.11.0 MPS (Apple M3 Pro)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/dataset_split/data.yaml, degrees=5.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=coins_v4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspective=0.0, plots

Overriding model.yaml nc=80 with nc=5



                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


...

                    50          1          1      0.942          1      0.995      0.796


                   500          2          2      0.798          1      0.995      0.671


Speed: 0.2ms preprocess, 101.0ms inference, 0.0ms loss, 132.6ms postprocess per image


Results saved to /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/runs/detect/coins_v4


Entrenamiento finalizado.


## 2. Persistencia del mejor checkpoint en `model/`

In [3]:
best_pt = RUNS_DIR / "detect" / RUN_NAME / "weights" / "best.pt"
last_pt = RUNS_DIR / "detect" / RUN_NAME / "weights" / "last.pt"
assert best_pt.exists(), f"No se encontró {best_pt}"
shutil.copy2(best_pt, MODEL_DIR / "best.pt")
if last_pt.exists():
    shutil.copy2(last_pt, MODEL_DIR / "last.pt")
print("Pesos copiados a:", MODEL_DIR)
for p in sorted(MODEL_DIR.iterdir()):
    print("  -", p.name, f"({p.stat().st_size / 1024:.0f} KB)")

Pesos copiados a: /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/model
  - best.pt (6109 KB)
  - last.pt (6109 KB)


## 3. Métricas finales sobre VALID y TEST

In [4]:
best_model = YOLO(str(MODEL_DIR / "best.pt"))

def metrics_to_row(m, split_name):
    return {
        "split": split_name,
        "precision": float(m.box.mp),
        "recall":    float(m.box.mr),
        "mAP@0.5":   float(m.box.map50),
        "mAP@0.5:0.95": float(m.box.map),
    }

val_metrics = best_model.val(data=str(DATA_YAML), split="val",  device=DEVICE,
                              project=str(RUNS_DIR / "detect"), name=f"{RUN_NAME}_val",  exist_ok=True, plots=True)
test_metrics = best_model.val(data=str(DATA_YAML), split="test", device=DEVICE,
                              project=str(RUNS_DIR / "detect"), name=f"{RUN_NAME}_test", exist_ok=True, plots=True)

df_metrics = pd.DataFrame([metrics_to_row(val_metrics, "valid"),
                            metrics_to_row(test_metrics, "test")]).set_index("split")
df_metrics.to_csv(REPORTS / "metrics_summary.csv")
df_metrics.round(4)

Ultralytics 8.4.47 🚀 Python-3.12.7 torch-2.11.0 MPS (Apple M3 Pro)


Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1536.3±277.0 MB/s, size: 38.1 KB)


val: Scanning /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/dataset_split/valid/labels.cache... 8 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8/8 11.2Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 7.1it/s 0.1s

...

                    50          2          2       0.48       0.48      0.448      0.333


                   500          2          2          1      0.823      0.995      0.745


Speed: 0.2ms preprocess, 4.7ms inference, 0.0ms loss, 79.2ms postprocess per image


Results saved to /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/runs/detect/coins_v4_test


,precision,recall,mAP@0.5,mAP@0.5:0.95
split,,,,
valid,0.7880,0.8000,0.9686,0.7632
test,0.6474,0.6607,0.8855,0.6235


## 4. Métricas por clase (sobre VALID)

In [5]:
names = best_model.names
p_cls = val_metrics.box.p
r_cls = val_metrics.box.r
ap50 = val_metrics.box.ap50
ap   = val_metrics.box.ap.mean(axis=1) if val_metrics.box.ap.ndim > 1 else val_metrics.box.ap
ids  = val_metrics.box.ap_class_index

df_cls = pd.DataFrame({
    "class": [names[int(i)] for i in ids],
    "precision": np.round(p_cls, 4),
    "recall":    np.round(r_cls, 4),
    "mAP@0.5":   np.round(ap50, 4),
    "mAP@0.5:0.95": np.round(ap, 4),
})
df_cls.to_csv(REPORTS / "metrics_per_class_valid.csv", index=False)
df_cls

,class,precision,recall,mAP@0.5,mAP@0.5:0.95
0,100,0.5435,1.0,0.863,0.7569
1,1000,0.6560,1.0,0.995,0.7960
2,200,1.0000,0.0,0.995,0.7960
3,50,0.9420,1.0,0.995,0.7960
4,500,0.7985,1.0,0.995,0.6712


## 5. Inferencia de ejemplo sobre el split TEST

Tomamos hasta 8 imágenes de test y mostramos detecciones con desglose monetario (sumatoria de denominaciones).

In [6]:
import yaml
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
_names = cfg["names"]
if isinstance(_names, dict):
    _names = [_names[i] for i in sorted(_names)]
VALUE_COP = {i: (int(n) if str(n).isdigit() else 0) for i, n in enumerate(_names)}

test_dir = ROOT / "dataset_split" / "test" / "images"
test_imgs = sorted(test_dir.iterdir())[:8]
n = len(test_imgs)
cols = 2
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))
axes = np.array(axes).reshape(-1)
for ax, img_path in zip(axes, test_imgs):
    pred = best_model.predict(source=str(img_path), conf=0.25, device=DEVICE, verbose=False)[0]
    counts = {}
    total_cop = 0
    for cls_id in pred.boxes.cls.cpu().numpy().astype(int):
        nm = pred.names[int(cls_id)]
        counts[nm] = counts.get(nm, 0) + 1
        total_cop += VALUE_COP.get(int(cls_id), 0)
    annotated = pred.plot()
    ax.imshow(annotated[:, :, ::-1])
    summary = " · ".join(f"{k}: {v}" for k, v in sorted(counts.items())) or "(sin detecciones)"
    ax.set_title(f"{img_path.name[:30]}\n{summary}\nTOTAL = {total_cop} COP", fontsize=9)
    ax.axis("off")
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(REPORTS / "test_inference.png", dpi=120)
plt.show()

<Figure size 1200x2000 with 8 Axes>

## 6. Resumen ejecutivo

Las celdas anteriores generaron:
- `model/best.pt` — pesos finales del modelo (axis-aligned bboxes, 5 clases).
- `reports/metrics_summary.csv` y `reports/metrics_per_class_valid.csv`.
- `reports/test_inference.png` con predicciones de ejemplo.
- `runs/detect/coins_v4/` con curvas de pérdida, matriz de confusión, etc.

Próxima fase: exportar `best.pt` a un formato apto para inferencia móvil (ONNX o TFLite) y arrancar la app Ionic + React + Capacitor.